<a href="https://colab.research.google.com/github/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego/blob/integration%2Fmodelos/notebooks/02_entrenamiento_YOLOv8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Entrenamiento de modelos YOLO para detección de humo y fuego

- Prueba iniciaal con yolov8n
## Salidas esperadas

El entrenamiento genera:

- pesos del modelo (`best.pt` y `last.pt`);
- métricas de entrenamiento y validación;
- curvas de desempeño;
- matriz de confusión;
- carpeta de resultados asociada al experimento.

In [37]:
# ============================================================
# Setup general del entorno
# ============================================================

from pathlib import Path
import os
import sys
import random
import shutil
import time
import yaml

SEED = 42
random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules

print("Ejecutando en Google Colab:", IN_COLAB)

Ejecutando en Google Colab: True


In [38]:
# ============================================================
# Repositorio
# ============================================================

REPO_URL = "https://github.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego"
REPO_NAME = "VpC2---Deteccion-de-humo-y-fuego"
TARGET_BRANCH = "integration/modelos"

PROJECT_DIR = Path("/content") / REPO_NAME

if IN_COLAB:
    if not PROJECT_DIR.exists():
        %cd /content
        !git clone --branch {TARGET_BRANCH} --single-branch {REPO_URL}.git

    %cd {PROJECT_DIR}
else:
    PROJECT_DIR = Path.cwd()

print("Proyecto:", PROJECT_DIR)
print("Branch:")
!git branch --show-current

/content/VpC2---Deteccion-de-humo-y-fuego
Proyecto: /content/VpC2---Deteccion-de-humo-y-fuego
Branch:
integration/modelos


In [25]:
# ============================================================
# Instalación de dependencias
# ============================================================

REPO_URL = "https://github.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego"
REPO_NAME = "VpC2---Deteccion-de-humo-y-fuego"

if IN_COLAB:
    !pip install -q -r https://raw.githubusercontent.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego/main/requirements.txt

print("Dependencias instaladas.")

Dependencias instaladas.


In [27]:
# ============================================================
# Verificación de GPU
# ============================================================

import torch

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No se detectó GPU. Para entrenar se recomienda activar GPU en Colab.")

CUDA disponible: True
GPU: Tesla T4


In [28]:
# ============================================================
# Montar Google Drive
# ============================================================

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/VCII_DFire")
DRIVE_RUNS_DIR = DRIVE_PROJECT_DIR / "runs"
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta principal en Drive:", DRIVE_PROJECT_DIR)
print("Carpeta de corridas:", DRIVE_RUNS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carpeta principal en Drive: /content/drive/MyDrive/VCII_DFire
Carpeta de corridas: /content/drive/MyDrive/VCII_DFire/runs


In [29]:
# ============================================================
# Carga de configuración del experimento
# ============================================================

EXPERIMENT_CONFIG_PATH = PROJECT_DIR / "configs" / "experiments" / "yolov8n_baseline.yaml"

if not EXPERIMENT_CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración: {EXPERIMENT_CONFIG_PATH}"
    )

with open(EXPERIMENT_CONFIG_PATH, "r", encoding="utf-8") as file:
    experiment_config = yaml.safe_load(file)

experiment_name = experiment_config["experiment"]["name"]
model_name = experiment_config["experiment"]["model"]
family = experiment_config["experiment"]["family"]

print("Experimento:", experiment_name)
print("Familia:", family)
print("Modelo:", model_name)
print("Descripción:", experiment_config["experiment"]["description"])
print("Config path:", EXPERIMENT_CONFIG_PATH)

Experimento: yolov8n_baseline
Familia: YOLOv8
Modelo: yolov8n.pt
Descripción: YOLOv8n entrenado durante 50 épocas sobre D-Fire.
Config path: /content/VpC2---Deteccion-de-humo-y-fuego/configs/experiments/yolov8n_baseline.yaml


In [30]:
# ============================================================
# Descarga o localización del dataset
# ============================================================

import kagglehub

DATASET_ID = "sayedgamal99/smoke-fire-detection-yolo"

dataset_root = Path(kagglehub.dataset_download(DATASET_ID))

print("Dataset descargado/localizado en:")
print(dataset_root)

print("\nContenido del directorio raíz del dataset:")
for item in dataset_root.iterdir():
    print("-", item)

Using Colab cache for faster access to the 'smoke-fire-detection-yolo' dataset.
Dataset descargado/localizado en:
/kaggle/input/smoke-fire-detection-yolo

Contenido del directorio raíz del dataset:
- /kaggle/input/smoke-fire-detection-yolo/data.yaml
- /kaggle/input/smoke-fire-detection-yolo/data


In [31]:
# ============================================================
# Detección estructura del dataset
# ============================================================

def find_yolo_dataset_dir(root: Path) -> Path:
    """
    Busca automáticamente la carpeta que contiene la estructura esperada:
    train/images, train/labels, val/images, val/labels.
    """
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]

    for candidate in candidates:
        train_images = candidate / "train" / "images"
        train_labels = candidate / "train" / "labels"
        val_images = candidate / "val" / "images"
        val_labels = candidate / "val" / "labels"

        if (
            train_images.exists()
            and train_labels.exists()
            and val_images.exists()
            and val_labels.exists()
        ):
            return candidate

    raise FileNotFoundError(
        "No se encontró una estructura YOLO válida con train/images, train/labels, val/images y val/labels."
    )

DATA_DIR = find_yolo_dataset_dir(dataset_root)

print("Carpeta de datos YOLO detectada:")
print(DATA_DIR)

for split in ["train", "val", "test"]:
    split_dir = DATA_DIR / split
    print(f"{split}: existe={split_dir.exists()} -> {split_dir}")

Carpeta de datos YOLO detectada:
/kaggle/input/smoke-fire-detection-yolo/data
train: existe=True -> /kaggle/input/smoke-fire-detection-yolo/data/train
val: existe=True -> /kaggle/input/smoke-fire-detection-yolo/data/val
test: existe=True -> /kaggle/input/smoke-fire-detection-yolo/data/test


In [32]:
# ============================================================
# Creación del archivo YAML para YOLO
# ============================================================

DFIRE_YAML = Path("/content/dfire_colab.yaml")

dfire_config = {
    "path": str(DATA_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 2,
    "names": {
        0: "smoke",
        1: "fire",
    },
}

with open(DFIRE_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(dfire_config, file, sort_keys=False, allow_unicode=True)

print("Archivo YAML creado en:")
print(DFIRE_YAML)

print("\nContenido:")
with open(DFIRE_YAML, "r", encoding="utf-8") as file:
    print(file.read())

Archivo YAML creado en:
/content/dfire_colab.yaml

Contenido:
path: /kaggle/input/smoke-fire-detection-yolo/data
train: train/images
val: val/images
test: test/images
nc: 2
names:
  0: smoke
  1: fire



In [33]:
# # ============================================================
# validación rápida del dataset
# ============================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_split_files(data_dir: Path, split:str):
  images_dir = data_dir / split / "images"
  labels_dir = data_dir / split / "labels"

  image_files = [f for f in images_dir.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS]
  label_files = [f for f in labels_dir.iterdir() if f.suffix.lower() == ".txt"]

  return len(image_files), len(label_files)

for split in ["train", "val", "test"]:
  images_count, labels_count = count_split_files(DATA_DIR, split)
  print(f"{split:5s} | imágenes: {images_count:6d} | etiquetas: {labels_count:6d}")




train | imágenes:  14122 | etiquetas:  14122
val   | imágenes:   3099 | etiquetas:   3099
test  | imágenes:   4306 | etiquetas:   4306


In [35]:
# ============================================================
# Función de entrenamiento desde configuración
# ============================================================

from ultralytics import YOLO
import pandas as pd

def train_from_config(config: dict, data_yaml: Path) -> dict:
    """
    Entrena un modelo YOLO a partir de un archivo de configuración YAML.

    Los resultados se guardan en una carpeta específica por experimento.
    Se almacena un checkpoint por época para preservar el historial
    del entrenamiento.
    """

    exp = config["experiment"]
    train_cfg = config["training"]
    output_cfg = config["output"]

    experiment_name = exp["name"]
    model_name = exp["model"]
    project_dir = Path(output_cfg["project"])

    project_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print(f"Experimento: {experiment_name}")
    print(f"Familia: {exp['family']}")
    print(f"Modelo: {model_name}")
    print(f"Resultados en: {project_dir / experiment_name}")
    print("=" * 80)

    start_time = time.time()

    # --------------------------------------------------------
    # Modelo preentrenado
    # --------------------------------------------------------

    model = YOLO(model_name)

    # --------------------------------------------------------
    # Entrenamiento
    # --------------------------------------------------------

    results = model.train(
        data=str(data_yaml),

        epochs=train_cfg["epochs"],
        imgsz=train_cfg["imgsz"],
        batch=train_cfg["batch"],
        patience=train_cfg["patience"],

        optimizer=train_cfg["optimizer"],
        lr0=train_cfg["lr0"],

        seed=train_cfg["seed"],

        project=str(project_dir),
        name=experiment_name,

        save=True,

        # Guarda un checkpoint independiente cada epoch
        save_period=train_cfg.get("save_period", 1),

        # Evita reutilizar silenciosamente una corrida anterior
        exist_ok=False,

        plots=True,
    )

    elapsed_time = time.time() - start_time

    # --------------------------------------------------------
    # Rutas de resultados
    # --------------------------------------------------------

    experiment_dir = Path(results.save_dir)

    best_model_path = experiment_dir / "weights" / "best.pt"
    last_model_path = experiment_dir / "weights" / "last.pt"
    results_csv = experiment_dir / "results.csv"

    # --------------------------------------------------------
    # Resumen del experimento
    # --------------------------------------------------------

    summary = {
        "experiment": experiment_name,
        "family": exp["family"],
        "model": model_name,

        "epochs": train_cfg["epochs"],
        "imgsz": train_cfg["imgsz"],
        "batch": train_cfg["batch"],

        "training_time_min": round(elapsed_time / 60, 2),

        "experiment_dir": str(experiment_dir),
        "best_model_path": str(best_model_path),
        "last_model_path": str(last_model_path),
        "results_csv": str(results_csv),

        "best_exists": best_model_path.exists(),
        "last_exists": last_model_path.exists(),
    }

    print("\nEntrenamiento finalizado.")
    print("Tiempo total [min]:", summary["training_time_min"])
    print("Best model:", best_model_path)
    print("Last model:", last_model_path)

    return summary

In [36]:
# ============================================================
# Ejecución opcional del entrenamiento
# ============================================================

RUN_TRAINING = False

if RUN_TRAINING:
    experiment_summary = train_from_config(
        config=experiment_config,
        data_yaml=DFIRE_YAML,
    )

    display(pd.DataFrame([experiment_summary]))

else:
    print("Entrenamiento omitido.")
    print("Se utilizarán los artefactos del experimento ya entrenado.")

Entrenamiento omitido.
Se utilizarán los artefactos del experimento ya entrenado.


## Resultados del experimento final

Para el análisis se utiliza la corrida final de YOLOv8n entrenada durante 50 épocas. A partir de los artefactos almacenados en Google Drive se recuperan la configuración, las métricas y los resultados necesarios para la comparación entre modelos.


In [39]:
# ============================================================
# Artefactos del experimento final
# ============================================================

FINAL_RUN_DIR = DRIVE_RUNS_DIR / "yolov8n_baseline-2"

ARGS_PATH = FINAL_RUN_DIR / "args.yaml"
RESULTS_CSV = FINAL_RUN_DIR / "results.csv"
BEST_PT = FINAL_RUN_DIR / "weights" / "best.pt"

required_files = [ARGS_PATH, RESULTS_CSV, BEST_PT]
missing = [path for path in required_files if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Faltan artefactos del experimento:\n"
        + "\n".join(str(path) for path in missing)
    )

with open(ARGS_PATH, "r", encoding="utf-8") as file:
    final_args = yaml.safe_load(file)

history = pd.read_csv(RESULTS_CSV)
history.columns = history.columns.str.strip()

print("Corrida:", FINAL_RUN_DIR.name)
print("Épocas configuradas:", final_args["epochs"])
print("Épocas registradas:", len(history))
print("imgsz:", final_args["imgsz"])
print("batch:", final_args["batch"])
print("best.pt:", BEST_PT.exists())

if int(final_args["epochs"]) != 50 or len(history) != 50:
    raise RuntimeError(
        "La corrida seleccionada no corresponde al experimento final de 50 épocas."
    )

Corrida: yolov8n_baseline-2
Épocas configuradas: 50
Épocas registradas: 50
imgsz: 640
batch: 16
best.pt: True


In [40]:
# ============================================================
# Metadata del entrenamiento
# ============================================================

from ultralytics import YOLO
import pandas as pd

model = YOLO(str(BEST_PT))

params_M = sum(
    parameter.numel()
    for parameter in model.model.parameters()
) / 1e6

best_idx = history["metrics/mAP50-95(B)"].idxmax()
best_row = history.loc[best_idx]
best_epoch = int(best_row["epoch"])

times = history["time"].astype(float).to_numpy()

total_seconds = times[0]

for i in range(1, len(times)):
    delta = times[i] - times[i - 1]

    if delta >= 0:
        total_seconds += delta
    else:
        # El contador se reinició al reanudar el entrenamiento
        total_seconds += times[i]

train_time_min = total_seconds / 60

print(f"Modelo: YOLOv8n")
print(f"Parámetros: {params_M:.2f} M")
print(f"Épocas: {final_args['epochs']}")
print(f"Mejor época según mAP@0.5:0.95: {best_epoch}")
print(f"Tiempo registrado de entrenamiento: {train_time_min:.2f} min")

Modelo: YOLOv8n
Parámetros: 3.01 M
Épocas: 50
Mejor época según mAP@0.5:0.95: 50
Tiempo registrado de entrenamiento: 243.23 min


## Evaluación del modelo final

Se evalúa el mejor checkpoint sobre el conjunto de validación de D-Fire para obtener métricas globales, métricas por clase y velocidad de inferencia.

In [41]:
# ============================================================
# Evaluación sobre el conjunto de validación
# ============================================================

device = 0 if torch.cuda.is_available() else "cpu"

metrics = model.val(
    data=str(DFIRE_YAML),
    split="val",
    imgsz=int(final_args["imgsz"]),
    batch=int(final_args["batch"]),
    device=device,
    plots=True,
    project=str(DRIVE_RUNS_DIR),
    name="yolov8n_baseline_val_final",
    exist_ok=True,
)

print("Resultados de validación:")
print(metrics.save_dir)

Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 29.4±13.3 MB/s, size: 227.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /kaggle/input/smoke-fire-detection-yolo/data/val/labels... 3094 images, 1375 backgrounds, 5 corrupt: 100% ━━━━━━━━━━━━ 3099/3099 766.5it/s 4.0s
val: /kaggle/input/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg'
val: /kaggle/input/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg'
val: /kaggle/input/s

In [42]:
# ============================================================
# Métricas por clase
# ============================================================

class_metrics = pd.DataFrame(
    metrics.summary(decimals=5)
)

display(class_metrics)

,Class,Images,Instances,Box-P,Box-R,Box-F1,mAP50,mAP50-95
0,smoke,1545,1751,0.81845,0.75100,0.78328,0.82296,0.51412
1,fire,875,2166,0.73899,0.63527,0.68322,0.71992,0.38095


In [43]:
# ============================================================
# Resumen estandarizado del experimento
# ============================================================

results_dict = metrics.results_dict

precision = float(results_dict["metrics/precision(B)"])
recall = float(results_dict["metrics/recall(B)"])

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall > 0
    else 0.0
)

class_results = class_metrics.set_index("Class")

smoke = class_results.loc["smoke"]
fire = class_results.loc["fire"]

inference_ms = float(metrics.speed["inference"])

fps = (
    1000.0 / inference_ms
    if inference_ms > 0
    else float("nan")
)

device_name = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

summary = pd.DataFrame([{
    "experiment": experiment_name,
    "family": family,
    "model": model_name,

    "params_M": round(params_M, 2),
    "epochs": int(final_args["epochs"]),
    "imgsz": int(final_args["imgsz"]),
    "batch": int(final_args["batch"]),
    "train_time_min": round(train_time_min, 2),

    "mAP50": float(results_dict["metrics/mAP50(B)"]),
    "mAP50_95": float(results_dict["metrics/mAP50-95(B)"]),

    "precision": precision,
    "recall": recall,
    "f1": f1,

    "mAP50_smoke": float(smoke["mAP50"]),
    "mAP50_fire": float(fire["mAP50"]),

    "mAP50_95_smoke": float(smoke["mAP50-95"]),
    "mAP50_95_fire": float(fire["mAP50-95"]),

    "fps": round(fps, 2),
    "device": device_name,
    "split": "val",

    "best_epoch": best_epoch,
}])

display(summary)

,experiment,family,model,params_M,epochs,imgsz,batch,train_time_min,mAP50,mAP50_95,...,recall,f1,mAP50_smoke,mAP50_fire,mAP50_95_smoke,mAP50_95_fire,fps,device,split,best_epoch
0,yolov8n_baseline,YOLOv8,yolov8n.pt,3.01,50,640,16,243.23,0.771442,0.447535,...,0.693136,0.733439,0.82296,0.71992,0.51412,0.38095,454.4,Tesla T4,val,50


In [44]:
# ============================================================
# Exportación de resultados
# ============================================================

REPORTS_RESULTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "results"
    / experiment_name
)

REPORTS_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Resumen para comparación
summary.to_csv(
    REPORTS_RESULTS_DIR / "metrics_summary.csv",
    index=False,
)

# Artefactos del entrenamiento original
for filename in [
    "results.csv",
    "results.png",
    "args.yaml",
]:
    src = FINAL_RUN_DIR / filename

    if src.exists():
        shutil.copy2(
            src,
            REPORTS_RESULTS_DIR / filename,
        )

# Figuras producidas al evaluar best.pt
VALIDATION_DIR = Path(metrics.save_dir)

for filename in [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "BoxF1_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
]:
    src = VALIDATION_DIR / filename

    if src.exists():
        shutil.copy2(
            src,
            REPORTS_RESULTS_DIR / filename,
        )

# Configuración utilizada
shutil.copy2(
    EXPERIMENT_CONFIG_PATH,
    REPORTS_RESULTS_DIR / "experiment_config_used.yaml",
)

print("Resultados guardados en:")
print(REPORTS_RESULTS_DIR)

print("\nArchivos:")
for path in sorted(REPORTS_RESULTS_DIR.iterdir()):
    print("-", path.name)

Resultados guardados en:
/content/VpC2---Deteccion-de-humo-y-fuego/reports/results/yolov8n_baseline

Archivos:
- BoxF1_curve.png
- BoxPR_curve.png
- BoxP_curve.png
- BoxR_curve.png
- args.yaml
- confusion_matrix.png
- confusion_matrix_normalized.png
- experiment_config_used.yaml
- metrics_summary.csv
- results.csv
- results.png
